In [1]:
import pandas as pd
import numpy as np

path = "Dataset_TD4.xlsx"
df = pd.read_excel(path)

print(df.shape)
print(list(df.columns))
df.head()

(1001, 5)
['transaction date (1=1day=24 hours)', 'bid-ask spread', 'volume of the transaction (if known)', 'Sign of the transaction', 'Price (before transaction)']


,transaction date (1=1day=24 hours),bid-ask spread,volume of the transaction (if known),Sign of the transaction,Price (before transaction)
0,0.000202,0.1100,8.0,-1,100.000
1,0.001070,0.1030,NaN,1,99.984
2,0.001496,0.1015,NaN,-1,100.029
3,0.003336,0.0920,NaN,1,99.979
4,0.003952,0.1106,NaN,1,100.060


In [2]:
#on trie les données par date de transaction, identifie les prix avant transactions et le signe de la trnansaction
df = df.sort_values("transaction date (1=1day=24 hours)").reset_index(drop=True)
df["dp"] = df["Price (before transaction)"].shift(-1) - df["Price (before transaction)"] #la différence de prix avant transaction
df["eps_t"] = df["Sign of the transaction"]
df["eps_tm1"] = df["Sign of the transaction"].shift(1)
reg_df = df[["dp", "eps_t", "eps_tm1", "bid-ask spread"]].dropna()


In [3]:
print("reg_df shape:", reg_df.shape)
print("Unique eps_t:", sorted(reg_df["eps_t"].unique()))
print("Unique eps_tm1:", sorted(reg_df["eps_tm1"].unique()))
print("Spread min/max:", reg_df["bid-ask spread"].min(), reg_df["bid-ask spread"].max())
reg_df.head()

reg_df shape: (999, 4)
Unique eps_t: [-1, 1]
Unique eps_tm1: [-1.0, 1.0]
Spread min/max: 0.0693 0.1326


,dp,eps_t,eps_tm1,bid-ask spread
1,0.045,1,-1.0,0.1030
2,-0.050,-1,1.0,0.1015
3,0.081,1,-1.0,0.0920
4,0.100,1,1.0,0.1106
5,0.004,1,1.0,0.1028


We first reorganize the transaction-level data in order to match Bouchaud’s price impact framework. Prices are sorted chronologically and price variations are defined as dp = pt+1 - pt. We then construct the contemporaneous and lagged trade signs eps_t and eps_tm1 as well as the bid–ask spread, which are used as explanatory variables in the price impact model.

In [4]:
y = reg_df["dp"].values.reshape(-1, 1)
X = reg_df[["eps_t", "eps_tm1", "bid-ask spread"]].values
X = np.column_stack([np.ones(len(X)), X])

In [5]:
XtX = X.T @ X
XtX_inv = np.linalg.inv(XtX)
XtY = X.T @ y

beta_tilde = XtX_inv @ XtY
beta_tilde

array([[ 0.00168096],
       [ 0.06225754],
       [-0.00389847],
       [-0.0065425 ]])

In [6]:
y_tilde = X @ beta_tilde
resid = y - y_tilde

n, k = X.shape
sigma2 = (resid.T @ resid) / (n - k)

cov_beta = sigma2[0, 0] * XtX_inv
se = np.sqrt(np.diag(cov_beta)).reshape(-1, 1)

t_stats = beta_tilde / se

beta_tilde, se, t_stats

(array([[ 0.00168096],
        [ 0.06225754],
        [-0.00389847],
        [-0.0065425 ]]),
 array([[0.01207228],
        [0.00124056],
        [0.00124055],
        [0.11947543]]),
 array([[ 0.13924174],
        [50.18522   ],
        [-3.14252218],
        [-0.05476022]]))

In [7]:
ss_tot = np.sum((y - y.mean())**2)
ss_res = np.sum(resid**2)
r2 = 1 - ss_res / ss_tot
r2

0.7180233644323013

In [8]:
names = ["const", "lambda (eps_t)", "eta (eps_tm1)", "beta (spread)"]

print(f"N={n}, k={k}, R2={r2:.6f}\n")
for nm, b, s, t in zip(names, beta_tilde.flatten(), se.flatten(), t_stats.flatten()):
    print(f"{nm:16s} coef={b: .6e}   se={s:.2e}   t={t:.2f}")


N=999, k=4, R2=0.718023

const            coef= 1.680965e-03   se=1.21e-02   t=0.14
lambda (eps_t)   coef= 6.225754e-02   se=1.24e-03   t=50.19
eta (eps_tm1)    coef=-3.898470e-03   se=1.24e-03   t=-3.14
beta (spread)    coef=-6.542501e-03   se=1.19e-01   t=-0.05


In [9]:
# Core Bouchaud model (no spread)
y = reg_df["dp"].values.reshape(-1, 1)

Xc = reg_df[["eps_t", "eps_tm1"]].values
Xc = np.column_stack([np.ones(len(Xc)), Xc])

XtXc = Xc.T @ Xc
XtXc_inv = np.linalg.inv(XtXc)
beta_c = XtXc_inv @ (Xc.T @ y)

y_hat_c = Xc @ beta_c
resid_c = y - y_hat_c

n_c, k_c = Xc.shape
ss_tot = np.sum((y - y.mean())**2)
ss_res = np.sum(resid_c**2)
r2_c = 1 - ss_res/ss_tot

sigma2_c = (resid_c.T @ resid_c) / (n_c - k_c)
cov_beta_c = sigma2_c[0,0] * XtXc_inv
se_c = np.sqrt(np.diag(cov_beta_c)).reshape(-1,1)
t_c = beta_c / se_c

names_c = ["const", "lambda (eps_t)", "eta (eps_tm1)"]
print(f"[CORE] N={n_c}, k={k_c}, R2={r2_c:.6f}\n")
for nm, b, s, t in zip(names_c, beta_c.flatten(), se_c.flatten(), t_c.flatten()):
    print(f"{nm:16s} coef={b: .6e}   se={s:.2e}   t={t:.2f}")


[CORE] N=999, k=3, R2=0.718023

const            coef= 1.023382e-03   se=1.24e-03   t=0.83
lambda (eps_t)   coef= 6.225859e-02   se=1.24e-03   t=50.22
eta (eps_tm1)    coef=-3.899514e-03   se=1.24e-03   t=-3.15
